In [1]:
import polars as pl

In [2]:
features = pl.read_parquet("../data/feats_good_w_objid.parquet").drop("diaObjectForcedSource","_healpix_29")

In [3]:
import distclassipy as dcpy
from distclassipy.anomaly import DistanceAnomaly

In [4]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

In [5]:
models = {
    "IF": IsolationForest( #using alerce anomaly params
        n_estimators=100,
        max_samples=128,
        contamination=0.001,
        random_state=42,
    ),

    "LOF": LocalOutlierFactor(
        n_neighbors=20, 
        novelty=True,  # IMPORTANT: Allows use on new data
        contamination='auto'
    ),
    
    "OC-SVM": OneClassSVM(
        kernel='rbf',
        nu=0.001
    ),
}

In [6]:
for model_name, model in models.items():
    feat_vals = features.drop("diaObjectId").to_numpy()
    model.fit(feat_vals)
    scores = model.decision_function(feat_vals)
    if model_name in ["iForest", "LOF", "OC-SVM"]:
        scores = -scores # high = anomaly
    features = features.with_columns(pl.Series(model_name, scores))

In [8]:
features

diaObjectId,amplitude_g,amplitude_r,amplitude_i,amplitude_z,inter_percentile_range_25_g,inter_percentile_range_25_r,inter_percentile_range_25_i,inter_percentile_range_25_z,anderson_darling_normal_g,anderson_darling_normal_r,anderson_darling_normal_i,anderson_darling_normal_z,beyond_1_std_g,beyond_1_std_r,beyond_1_std_i,beyond_1_std_z,chi2_pvar_g,chi2_pvar_r,chi2_pvar_i,chi2_pvar_z,cusum_g,cusum_r,cusum_i,cusum_z,eta_g,eta_r,eta_i,eta_z,eta_e_g,eta_e_r,eta_e_i,eta_e_z,excess_variance_g,excess_variance_r,excess_variance_i,excess_variance_z,…,weighted_mean_i,weighted_mean_z,villar_fit_amplitude_g,villar_fit_baseline_g,villar_fit_reference_time_g,villar_fit_rise_time_g,villar_fit_fall_time_g,villar_fit_plateau_rel_amplitude_g,villar_fit_plateau_duration_g,villar_fit_reduced_chi2_g,villar_fit_amplitude_r,villar_fit_baseline_r,villar_fit_reference_time_r,villar_fit_rise_time_r,villar_fit_fall_time_r,villar_fit_plateau_rel_amplitude_r,villar_fit_plateau_duration_r,villar_fit_reduced_chi2_r,villar_fit_amplitude_i,villar_fit_baseline_i,villar_fit_reference_time_i,villar_fit_rise_time_i,villar_fit_fall_time_i,villar_fit_plateau_rel_amplitude_i,villar_fit_plateau_duration_i,villar_fit_reduced_chi2_i,villar_fit_amplitude_z,villar_fit_baseline_z,villar_fit_reference_time_z,villar_fit_rise_time_z,villar_fit_fall_time_z,villar_fit_plateau_rel_amplitude_z,villar_fit_plateau_duration_z,villar_fit_reduced_chi2_z,IF,LOF,OC-SVM
i64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64
786345564856911058,2.176949,0.078133,3.010708,0.03297,0.051931,0.023737,0.016369,0.014935,26.402351,1.074485,52.914322,0.330048,0.038462,0.226994,0.006711,0.32,0.0,0.0,0.0,0.014411,0.121229,0.098233,0.081249,0.09103,2.065254,1.698147,2.010862,2.096568,6.8959336e7,1.0359182e7,2.0568616e7,125083.367188,0.001037,2.9744e-7,0.000563,8.5827e-8,…,19.362778,19.054802,0.05664,21.655436,2671.679688,276.457306,220.695435,0.88386,35.202572,3.636694,0.12298,20.003065,2941.881836,127.238266,78.857437,0.994101,265.416473,1.513749,0.050309,19.353661,3312.279541,241.967911,752.414734,0.079632,378.15213,2.121732,0.02552,19.05393,2794.812256,17.984272,2.709798,0.532572,13.635175,0.701398,0.073135,-0.154973,-0.000756
786345358698480731,0.187675,1.181005,1.934206,0.252655,0.038715,0.062878,0.104319,0.091351,12.046857,25.299244,25.633566,0.258433,0.076923,0.086957,0.04698,0.29,0.0,0.0,0.0,0.651891,0.235516,0.081476,0.111284,0.16005,0.452247,1.856062,1.750109,2.017465,4.815348e6,1.3324657e7,1.0047813e7,127331.054688,0.000013,0.000074,0.000085,-0.000003,…,21.427994,21.414377,0.837674,21.512392,3017.364258,46.339844,218.744278,0.692249,42.526939,1.319993,0.77664,21.676386,3037.862061,6.451549,284.943085,0.031772,256.241821,3.632695,1.017414,20.943117,2881.125977,241.914871,216.592407,0.116851,192.092819,0.713692,0.342966,21.174513,2807.002441,4.655171,9.751328,0.228767,15.281721,0.461308,0.204944,-0.44986,-0.006739
786345427417957368,0.226052,0.2947,0.740931,0.304376,0.053337,0.060141,0.10284,0.080561,10.406733,7.945403,8.29187,1.107226,0.088608,0.133333,0.248322,0.264706,0.0,0.0,0.0,0.500011,0.234565,0.168787,0.182189,0.127026,0.458366,0.845379,1.45627,1.765759,6795562.5,4271748.5,1.1673459e7,108764.015625,0.000016,0.000004,0.000012,5.0202e-7,…,21.262669,21.362608,0.255959,21.674547,2978.499756,183.271408,19.335627,0.902626,38.447147,0.817191,0.369839,21.276978,2871.733154,131.133606,72.240616,0.58553,32.994495,0.644434,0.931735,20.905279,2447.601807,45.26606,219.636322,0.109319,225.081482,1.130591,0.335411,21.209419,2814.480225,3.83699,12.517359,0.222229,2.894141,0.475354,0.212424,-0.508171,-0.008851
786345358698481277,0.431983,0.530574,0.952628,0.99682,0.101589,0.146433,0.232765,0.21743,7.640248,3.622436,4.242921,2.796489,0.113924,0.2,0.198529,0.168317

In [9]:
features.write_parquet("../data/feats_good_w_objid_AD.parquet")